# 🧠 Day 3 — LLM APIs Explained (Hands-On, with Ollama)
### CodeLucky · 12-Day Programme on Modern AI, Generative AI & Agentic Systems

> **Who this is for:** Complete beginners to APIs. **Promise:** follow every cell in order and you'll be talking to your own local AI in ~15 minutes. **Cost:** ₹0 / $0 — everything runs on your own computer, offline.

> ⚠️ **Note on environment:** Unlike Day 1–2, this notebook runs **locally on your own machine** (or any machine with Ollama installed), *not* in Google Colab — Colab cannot run a persistent local server like Ollama. Install Ollama first (Step 1 below), then run this notebook with a local Jupyter install, or open a terminal alongside it.

**By the end you will be able to:**
- Explain what an API is in one sentence
- Install Ollama and run a real AI model on your own PC
- Send a request to the model with both `curl` and Python
- Read and understand the JSON going back and forth
- Control the model with `temperature`, `stream`, and chat history
- Build a tiny working chatbot


## 🍽️ The One Idea: an API Is a Waiter

| Restaurant | Software world |
|---|---|
| 🧑 You, the customer | Your app / your code |
| 🧑‍🍳 The waiter | The **API** |
| 🔥 The kitchen | The **LLM** (the AI model) |

You don't walk into the kitchen and cook — you tell the waiter what you want, the waiter takes it to the kitchen, and brings back your food.

> **📌 API** = a messenger that lets two programs talk using rules they both agree on. You send a **request**, you get a **response**.

## 💬 What an *LLM* API Actually Is
You send text → it sends text back.

**Request fields:** `model` (which AI brain), `prompt` (your question), `options` (extra settings, e.g. creativity).
**Response fields:** `response` (the generated text), `model`, `done` (finished?), token counts.

```json
{
  "model": "llama3.2:3b",
  "prompt": "Why is the sky blue?",
  "stream": false
}
```

**The round trip:** your code → POST JSON → Ollama server (`localhost:11434`) → the model generates text → JSON response back to your code. `11434` is Ollama's default "door number" (port).


## 🧩 Which Model Should I Use?

| Model | Pull command | Size | Best for |
|---|---|---|---|
| ⭐ `llama3.2:3b` *(MAIN PICK)* | `ollama pull llama3.2:3b` | ~2 GB | Start here. Best speed/quality balance. Runs on 8GB RAM. |
| 💾 `llama3.2:1b` *(LOW-RAM)* | `ollama pull llama3.2:1b` | ~1.3 GB | < 8 GB RAM or slow/no-GPU laptops |
| 🚀 `qwen3:4b` *(QUALITY)* | `ollama pull qwen3:4b` | ~2.5 GB | 8 GB+ free, want smarter replies |

**Everywhere below we use `llama3.2:3b`.** Swap the name in every command to use a different model.


---
## 🟢 Step 1 — Install Ollama

**Windows / macOS:** download the installer from https://ollama.com/download and run it.

**Linux:**
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

**Verify it installed** (in a terminal):
```bash
ollama --version
```
You should see a version number like `ollama version 0.x.x`.


## 📦 Step 2 — Download a Model
Downloads once (~2 GB) and caches on your PC forever after.
```bash
ollama pull llama3.2:3b
```
Low-RAM users: `ollama pull llama3.2:1b` instead.

Check what you've downloaded:
```bash
ollama list
```


## 💬 Step 3 — Chat Without Any Code (Sanity Check)
```bash
ollama run llama3.2:3b "Say hello in exactly 5 words"
```
Or open an interactive chat:
```bash
ollama run llama3.2:3b
```
Type `/bye` to exit. ✅ If this works, everything is set up correctly — the API steps below use the exact same model, just through code.


## 🌐 Step 4 — Your First API Call with `curl`
Keep Ollama running (already running in the background on Mac/Windows after install; on Linux run `ollama serve` in a separate terminal if needed).

```bash
curl http://localhost:11434/api/generate -d "{\"model\": \"llama3.2:3b\",\"prompt\": \"Why is the sky blue? Answer in 2 sentences.\",\"stream\": false}"
```

**Windows PowerShell:**
```powershell
curl http://localhost:11434/api/generate -Method POST -Body '{"model":"llama3.2:3b","prompt":"Why is the sky blue? Answer in 2 sentences.","stream":false}'
```

The answer comes back inside the `"response"` field of the JSON. 🎉


## 🐍 Step 5 — Your First API Call with Python
Same request, but from Python — how real apps do it. Make sure `requests` is installed: `pip install requests`.

In [ ]:
import requests  # the tool that sends web requests

# Send the request to our local Ollama "waiter"
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Why is the sky blue? Answer in 2 sentences.",
        "stream": False,   # False = get the whole answer at once
    },
)

data = response.json()
print(data["response"])

> The `curl` version and the Python version send the exact same JSON to the exact same address. The language doesn't matter — the API is the same. **That's the entire loop:** build a request → send it → read `response`. Everything else is a variation on this.

## 🔍 Understanding the JSON — Every Key Decoded

| Request key | Meaning |
|---|---|
| `"model"` | Which model answers — must be one you've pulled |
| `"prompt"` | Your question or instruction |
| `"stream"` | `false` = whole answer at once. `true` = piece by piece |
| `"options"` | Optional tuning, e.g. `{"temperature": 0.7}` |

| Response key | Meaning |
|---|---|
| `"response"` | ⭐ The generated text |
| `"done"` | `true` when finished |
| `"eval_count"` | Roughly how many tokens were generated |
| `"total_duration"` | How long it took (nanoseconds) |


## 🌊 Streaming Answers Word-by-Word
With `"stream": true`, the model sends the answer as it types — like ChatGPT does.

In [ ]:
import requests
import json

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Write a haiku about the ocean.",
        "stream": True,          # stream the answer piece by piece
    },
    stream=True,                 # tell requests to read it as a stream too
)

for line in response.iter_lines():
    if line:
        chunk = json.loads(line)
        print(chunk.get("response", ""), end="", flush=True)
        if chunk.get("done"):
            break
print()

## 🧵 Multi-Turn Chat (the Model Remembers)
`/api/generate` is one-shot. For real conversations, use `/api/chat` with a list of **messages**, each having a **role**: `system` (behavior/personality), `user` (human), `assistant` (AI's past replies).

In [ ]:
import requests

messages = [
    {"role": "system", "content": "You are a friendly tutor. Keep answers short."},
    {"role": "user", "content": "What is an API in one line?"},
]

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "llama3.2:3b",
        "messages": messages,
        "stream": False,
    },
)

reply = response.json()["message"]["content"]
print("AI:", reply)

> To continue the conversation, append the AI's reply to `messages` as an `assistant` message, then add the next `user` message, and call again. That's how "memory" works — **you** resend the history each time.

## 🌡️ The Temperature Dial
Controls how creative vs. focused the model is (roughly `0.0` to `1.0+`).

| Temperature | Behavior | Use for |
|---|---|---|
| `0.0` | 🎯 Focused, factual, repeatable | Math, code, exact facts |
| `0.4` | ⚖️ Balanced | General Q&A |
| `0.7` | 🎨 Creative | Brainstorming, writing |
| `1.0+` | 🎲 Wild, random | Poetry, weird ideas |


In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": "Give me a startup name for a coffee app.",
        "stream": False,
        "options": { "temperature": 0.9 },   # turn up the creativity
    },
)
print(response.json()["response"])
# Try running it with temperature 0.0 and then 1.0 and compare

## 🤖 Mini Project — a Tiny Terminal Chatbot
Combines everything: a chatbot that remembers the conversation.

*(Run this as a `.py` script in a terminal for the interactive `input()` loop to work — it won't run well inside a notebook cell.)*

In [ ]:
import requests

URL = "http://localhost:11434/api/chat"
MODEL = "llama3.2:3b"   # low-RAM users: change to "llama3.2:1b"

# Conversation history. The system message sets the personality.
messages = [
    {"role": "system", "content": "You are a helpful, concise assistant."}
]

print("🤖 Chatbot ready! Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ("quit", "exit", "bye"):
        print("👋 Goodbye!")
        break

    messages.append({"role": "user", "content": user_input})

    response = requests.post(
        URL,
        json={"model": MODEL, "messages": messages, "stream": False},
    )
    reply = response.json()["message"]["content"]

    print("AI:", reply, "\n")
    messages.append({"role": "assistant", "content": reply})

> 🏆 **You now understand LLM APIs end-to-end:** request/response, JSON, endpoints, streaming, chat history, temperature. Everything cloud APIs (OpenAI, Anthropic, Google) do is a variation on exactly this — same JSON shape, different address, plus an API key.

## 🛠️ Troubleshooting

| Problem | Fix |
|---|---|
| `command not found: ollama` | Reopen the terminal or restart your PC |
| `connection refused` on `localhost:11434` | Ollama isn't running — Linux: `ollama serve`; Mac/Windows: launch the app |
| `model 'x' not found` | Run `ollama pull llama3.2:3b` |
| Very slow / PC freezes | Model too big for RAM — switch to `llama3.2:1b`, close browser |
| `ModuleNotFoundError: requests` | `pip install requests` |
| `curl` errors on Windows | Use the PowerShell `-Method POST -Body '...'` version |
| Answer is empty | Read `response["response"]` for `/api/generate`, or `response["message"]["content"]` for `/api/chat` |

Check the server is alive:
```bash
curl http://localhost:11434
```
It should reply `Ollama is running`.


## 📝 Cheat Sheet

**Setup (once):**
```bash
ollama pull llama3.2:3b
ollama list
```

**Test without code:**
```bash
ollama run llama3.2:3b "Hi"
```

**One-shot API (generate):**
```
POST http://localhost:11434/api/generate
{ "model": "llama3.2:3b", "prompt": "...", "stream": false }
→ read  response["response"]
```

**Conversation API (chat):**
```
POST http://localhost:11434/api/chat
{ "model": "llama3.2:3b", "messages": [ {role, content}, ... ], "stream": false }
→ read  response["message"]["content"]
```

| Knob | Effect |
|---|---|
| `stream` | `false` = all at once · `true` = word-by-word |
| `temperature` | low = focused · high = creative |
| `messages` | send history → model "remembers" |

### 🎓 Where to Go Next
- Swap `llama3.2:3b` for `qwen3:4b` and compare answer quality.
- Point the same Python code at a **cloud** API (OpenAI/Anthropic) — only the URL and an API key change. The JSON shape is nearly identical.
- Build a small web UI, or feed the model your own documents (that's **RAG** — coming up in later days).

---
**CodeLucky · Faculty Development Programme**
